# Importing the libraries

In [2]:
import pandas as pd
import numpy as np

#### Load the prepared dataset

In [3]:
# Loading the prepared dataset created during data preparation.

df = pd.read_csv('../data/processed/prepared_data.csv')

In [4]:
df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
1,2022-01-01,S004,P0013,Groceries,West,136,104,385,20.24,10,Snowy,0,18.90,Winter,0,110
2,2022-01-01,S004,P0012,Electronics,West,111,111,113,118.15,0,Snowy,0,133.46,Winter,0,103
3,2022-01-01,S004,P0011,Clothing,West,195,60,293,52.89,0,Snowy,0,62.29,Winter,0,61
4,2022-01-01,S004,P0010,Groceries,West,223,120,597,30.02,0,Snowy,0,29.15,Winter,0,128


#### Convert Date back to datetime

In [6]:
# Converting the Date column to datetime format for date-based feature creation.

df['Date'] = pd.to_datetime(df['Date'])

In [8]:
# Checking the data types before creating new features.

df.dtypes

Date                  datetime64[ns]
Store ID                      object
Product ID                    object
Category                      object
Region                        object
Inventory Level                int64
Units Sold                     int64
Units Ordered                  int64
Price                        float64
Discount                       int64
Weather Condition             object
Promotion                      int64
Competitor Pricing           float64
Seasonality                   object
Epidemic                       int64
Demand                         int64
dtype: object

#### Date-Based Features

The original Date contains useful information, but a model can benefit from separate temporal features.

We'll create:

- Year
- Month
- Week
- Day
- Day of Week
- Weekend indicator

In [9]:
# Creating the year feature to capture yearly demand differences.

df['Year'] = df['Date'].dt.year

# Creating the month feature to capture monthly demand patterns.

df['Month'] = df['Date'].dt.month

# Creating the week feature to capture weekly seasonal patterns.

df['Week'] = df['Date'].dt.isocalendar().week.astype(int)

# Creating the day feature to capture daily patterns within each month.

df['Day'] = df['Date'].dt.day

# Creating the day-of-week feature to capture differences between weekdays.

df['Day_of_Week'] = df['Date'].dt.dayofweek

# Creating the weekend indicator to distinguish weekends from weekdays.

df['Is_Weekend'] = (df['Day_of_Week'] >= 5).astype(int)

In [10]:
# Displaying the newly created date-based features.

df[
    [
        'Date',
        'Year',
        'Month',
        'Week',
        'Day',
        'Day_of_Week',
        'Is_Weekend'
    ]
].head(10)

,Date,Year,Month,Week,Day,Day_of_Week,Is_Weekend
0,2022-01-01,2022,1,52,1,5,1
1,2022-01-01,2022,1,52,1,5,1
2,2022-01-01,2022,1,52,1,5,1
3,2022-01-01,2022,1,52,1,5,1
4,2022-01-01,2022,1,52,1,5,1
5,2022-01-01,2022,1,52,1,5,1
6,2022-01-01,2022,1,52,1,5,1
7,2022-01-01,2022,1,52,1,5,1
8,2022-01-01,2022,1,52,1,5,1
9,2022-01-01,2022,1,52,1,5,1


#### Inventory & Sales Features

Here we can create features that describe the relationship between inventory and actual sales.

In [11]:
# Creating the inventory-to-sales ratio to measure inventory available relative to units sold.

df['Inventory_to_Sales_Ratio'] = (
    df['Inventory Level'] /
    df['Units Sold'].replace(0, np.nan)
)

# Replacing undefined ratios caused by zero sales with zero.

df['Inventory_to_Sales_Ratio'] = (
    df['Inventory_to_Sales_Ratio'].fillna(0)
)

##### Inventory gap

In [12]:
# Creating the inventory gap to measure the difference between available inventory and units sold.

df['Inventory_Gap'] = (
    df['Inventory Level'] - df['Units Sold']
)

#### Pricing Features

The dataset contains both our product price and competitor pricing.

That's useful because relative pricing can potentially influence demand.

##### Price difference

In [13]:
# Creating the price difference to compare our product price with competitor pricing.

df['Price_Difference'] = (
    df['Price'] - df['Competitor Pricing']
)

##### Price difference percentage

In [14]:
# Creating the price difference percentage to measure our price relative to competitor pricing.

df['Price_Difference_Percentage'] = (
    (df['Price'] - df['Competitor Pricing']) /
    df['Competitor Pricing']
) * 100

#### Promotion & Discount Feature

We already have separate Promotion and Discount columns.

We can create a feature representing their combined effect.

##### Promotion discount interaction

In [15]:
# Creating an interaction feature to capture the combined effect of promotion and discount.

df['Promotion_Discount'] = (
    df['Promotion'] * df['Discount']
)

For example:

- Promotion = 0, Discount = 20 → 0
- Promotion = 1, Discount = 20 → 20

This allows the model to distinguish an actual promoted discount from the discount value alone

#### Lag Features

This is particularly important for a demand forecasting project.

Previous demand can provide information about current demand.

But we need to make sure the previous value comes from the same store and product.

##### Sorting the data

In [16]:
# Sorting the data by store, product, and date before creating lag features.

df = df.sort_values(
    ['Store ID', 'Product ID', 'Date']
).reset_index(drop=True)

##### Previous demand

In [18]:
# Creating the previous day's demand for each store-product combination.

df['Previous_Demand'] = (
    df.groupby(
        ['Store ID', 'Product ID']
    )['Demand']
    .shift(1)
)

##### Previous sales

In [19]:
# Creating the previous day's units sold for each store-product combination.

df['Previous_Units_Sold'] = (
    df.groupby(
        ['Store ID', 'Product ID']
    )['Units Sold']
    .shift(1)
)

#### Rolling Demand Features

Previous demand is useful, but an average of recent demand can provide a more stable signal.

We'll create a 7-day rolling demand average.

In [20]:
# Creating a seven-day rolling average of previous demand for each store-product combination.

df['Rolling_7_Day_Demand'] = (
    df.groupby(
        ['Store ID', 'Product ID']
    )['Demand']
    .transform(
        lambda x: x.shift(1).rolling(7).mean()
    )
)

Notice the .shift(1).

That's important.

We're using the previous seven days, not the current day's demand.

Otherwise, we'd leak the target into the feature.

#### Seven-day rolling sales

In [21]:
# Creating a seven-day rolling average of previous units sold for each store-product combination.

df['Rolling_7_Day_Sales'] = (
    df.groupby(
        ['Store ID', 'Product ID']
    )['Units Sold']
    .transform(
        lambda x: x.shift(1).rolling(7).mean()
    )
)

#### Check the Engineered Features

In [22]:
# Displaying the complete list of columns after feature engineering.

df.columns.tolist()

['Date',
 'Store ID',
 'Product ID',
 'Category',
 'Region',
 'Inventory Level',
 'Units Sold',
 'Units Ordered',
 'Price',
 'Discount',
 'Weather Condition',
 'Promotion',
 'Competitor Pricing',
 'Seasonality',
 'Epidemic',
 'Demand',
 'Year',
 'Month',
 'Week',
 'Day',
 'Day_of_Week',
 'Is_Weekend',
 'Inventory_to_Sales_Ratio',
 'Inventory_Gap',
 'Price_Difference',
 'Price_Difference_Percentage',
 'Promotion_Discount',
 'Previous_Demand',
 'Previous_Units_Sold',
 'Rolling_7_Day_Demand',
 'Rolling_7_Day_Sales']

In [23]:
# Inspecting the newly created features to verify their values.

df.head(10)

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,...,Is_Weekend,Inventory_to_Sales_Ratio,Inventory_Gap,Price_Difference,Price_Difference_Percentage,Promotion_Discount,Previous_Demand,Previous_Units_Sold,Rolling_7_Day_Demand,Rolling_7_Day_Sales
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,...,1,1.911765,93,-13.01,-15.175551,0,NaN,NaN,NaN,NaN
1,2022-01-02,S001,P0001,Electronics,North,93,71,0,65.63,5,...,1,1.309859,22,-8.03,-10.901439,0,115.0,102.0,NaN,NaN
2,2022-01-03,S001,P0001,Electronics,North,274,142,229,68.55,15,...,0,1.929577,132,-12.18,-15.087328,15,84.0,71.0,NaN,NaN
3,2022-01-04,S001,P0001,Electronics,North,132,42,0,61.66,10,...,0,3.142857,90,6.78,12.354227,0,132.0,142.0,NaN,NaN
4,2022-01-05,S001,P0001,Electronics,North,319,129,0,59.56,25,...,0,2.472868,190,2.22,3.871643,25,67.0,42.0,NaN,NaN
5,2022-01-06,S001,P0001,Electronics,North,190,98,308,84.03,0,...,0,1.938776,92,-11.51,-12.047310,0,110.0,129.0,NaN,NaN
6,2022-01-07,S001,P0001,Electronics,North,92,92,0,66.07,10,...,0,1.000000,0,2.50,3.932673,0,146.0,98.0,NaN,NaN
7,2022-01-08,S001,P0001,Electronics,North,308,96,248,68.56,10,...,1,3.208333,212,-12.43,-15.347574,10,87.0,92.0,105.857143,96.571429
8,2022-01-09,S001,P0001,Electronics,North,212,57,0,62.76,10,...,1,3.719298,155,-4.35,-6.481895,0,113.0,96.0,105.571429,95.714286
9,2022-01-10,S001,P0001,Electronics,North,403,88,0,68.99,5,...,0,4.579545,315,-12.15,-14.974119,0,87.0,57.0,106.000000,93.714286


#### Check Missing Values Created by Lag/Rolling Features

This is expected.

The first record for a store-product combination won't have a previous demand value, and the first several records won't have seven previous days.

In [24]:
# Checking missing values created by lag and rolling features.

df[
    [
        'Previous_Demand',
        'Previous_Units_Sold',
        'Rolling_7_Day_Demand',
        'Rolling_7_Day_Sales'
    ]
].isnull().sum()

Previous_Demand         100
Previous_Units_Sold     100
Rolling_7_Day_Demand    700
Rolling_7_Day_Sales     700
dtype: int64

#### Final Feature Validation

##### Check dataset information

In [25]:
# Checking the final dataset structure after feature engineering.

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76000 entries, 0 to 75999
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Date                         76000 non-null  datetime64[ns]
 1   Store ID                     76000 non-null  object        
 2   Product ID                   76000 non-null  object        
 3   Category                     76000 non-null  object        
 4   Region                       76000 non-null  object        
 5   Inventory Level              76000 non-null  int64         
 6   Units Sold                   76000 non-null  int64         
 7   Units Ordered                76000 non-null  int64         
 8   Price                        76000 non-null  float64       
 9   Discount                     76000 non-null  int64         
 10  Weather Condition            76000 non-null  object        
 11  Promotion                    76000 non-nu

In [26]:
# Checking the dataset shape after adding engineered features.

print("Dataset Shape:", df.shape)

Dataset Shape: (76000, 31)


#### Save the Engineered Dataset

In [27]:
# Saving the engineered dataset for model development.

df.to_csv(
    '../data/processed/engineered_data.csv',
    index=False
)

#### One important correction to our overall project

For this project, Demand is our target variable.

So when we eventually build the model, we need to make sure features such as:

- Demand
- Previous_Demand
- Rolling_7_Day_Demand

are handled correctly.

Demand itself will not be used as an input feature. The lagged and rolling versions are acceptable because they represent information that would have been available before the prediction date.

Also, I would not add dozens of artificial features just to make the project look more advanced. The features above have a clear business/forecasting purpose.
